In [ ]:
!pip install rdkit

# From Fingerprints To A Purchase Report

Last session turned a supplier's catalogue into seven clean structures, each with a row of descriptors and a drug-likeness score.
That table says how heavy, how greasy and how drug-like each compound is, but it cannot say which compounds *look alike*.
Today the same supplier offers nine more compounds, and the question is the one a purchasing chemist actually asks: which of them add new chemistry to what we already hold, and which are close relatives of compounds already on the shelf?

You will answer it with molecular fingerprints and [chemical similarity](https://en.wikipedia.org/wiki/Chemical_similarity), and finish with a report you could send to a colleague and one figure that shows the whole decision at a glance.

By the end the notebook prints something like this:

```
             name     nearest  tanimoto              band    qed
0    naproxen (R)    naproxen     1.000    close analogue  0.881
1      fenoprofen  ketoprofen     0.583  related scaffold  0.889
3   desloratadine  loratadine     0.579  related scaffold  0.795
4     terbutaline  salbutamol     0.553  related scaffold  0.629
2    flurbiprofen  ketoprofen     0.475  related scaffold  0.894
5  salicylic acid     aspirin     0.448  related scaffold  0.610
6      diflunisal     aspirin     0.250        background  0.861
7       celecoxib  salbutamol     0.123        background  0.754
8        caffeine  loratadine     0.101        background  0.538
```

and saves a figure of similarity against drug-likeness with one point per offered compound.


## Part 1 - Last Session's Catalogue, Rebuilt

A notebook remembers nothing between sessions, so the first few blocks rebuild last session's result before anything new is asked of it.
Everything today runs on [RDKit](https://www.rdkit.org/docs/) and [pandas](https://pandas.pydata.org/docs/), with [Matplotlib](https://matplotlib.org/stable/) for the figure at the end.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, Descriptors, Draw, MACCSkeys, QED
from rdkit.Chem.Draw import IPythonConsole

Below is the supplier's catalogue as it stood at the end of last session, with your repair to ketoprofen already made.

The two functions after it are also last session's.
`clean_catalogue` was handed to you then: it parses every record, reports the failures and keeps one row per structure.
`profile` is the function you wrote yourself in Part 3: it takes a name and a molecule and returns one dictionary of six descriptors.


In [ ]:
supplier_catalogue = {
    "naproxen (S)": "C[C@@H](c1ccc2cc(OC)ccc2c1)C(=O)O",
    "naproxen (R)": "C[C@H](c1ccc2cc(OC)ccc2c1)C(=O)O",
    "ibuprofen (racemate)": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "ibuprofen (S)": "CC(C)Cc1ccc(cc1)[C@H](C)C(=O)O",
    "ketoprofen": "CC(c1cccc(c1)C(=O)c1ccccc1)C(=O)O",
    "salbutamol": "CC(C)(C)NCC(O)c1ccc(O)c(CO)c1",
    "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "loratadine": "CCOC(=O)N1CCC(=C2c3ccc(Cl)cc3CCc3cccnc32)CC1",
    "amiodarone": "CCCCc1oc2ccccc2c1C(=O)c1cc(I)c(OCCN(CC)CC)c(I)c1",
}

In [ ]:
def clean_catalogue(catalogue, keep_stereo):
    """Parse every record in a catalogue, report failures, and keep only one row per structure

    :param catalogue: a dictionary of molecule names and their SMILES string
    :param keep_stereo: True or False - whether to keep the stereochemistry.
    :returns: a Pandas dataframe with the fixed data.
    """
    rows = []
    seen_smiles = set()
    for name, smiles in catalogue.items():
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Molecule {name:s} has invalid SMILES: '{smiles:s}'")
        else:
            canonical_smiles = Chem.MolToSmiles(mol, isomericSmiles=keep_stereo)
            if canonical_smiles in seen_smiles:
                print(f"Duplicate {name:s} skipped because it was stored earlier.")
            else:
                seen_smiles.add(canonical_smiles)
                rows.append({"name": name, "smiles": canonical_smiles, "mol": mol})

    print(f"Initial catalog had {len(catalogue):d} records. Cleaned dataset has {len(rows):d} unique structures.")
    return pd.DataFrame(rows)


def profile(name, mol):
    """Return a dictionary of six physicochemical descriptors for one molecule

    :param name: the name of the molecule
    :param mol: the rdkit mol object
    """
    return {"name": name,
            "mw"  : round(Descriptors.MolWt(mol), 1),
            "logp": round(Descriptors.MolLogP(mol), 2),
            "tpsa": round(Descriptors.TPSA(mol), 1),
            "hbd" : Descriptors.NumHDonors(mol),
            "hba" : Descriptors.NumHAcceptors(mol),
            "rotb": Descriptors.NumRotatableBonds(mol)}

#### Questions

1. Last session you argued that this catalogue should be stripped of its stereochemistry. Predict how many rows `dataset` will have and which two records `clean_catalogue` will report as duplicates, then run the block.


In [ ]:
dataset = clean_catalogue(supplier_catalogue, keep_stereo=False)

### Preparing the Table
---
Please note, that the fixes below uses [list-comprehensions](https://youtu.be/0MIv6ibqKso) extensively to write short code.

---

The code below (lines 1-2) makes the fix you argued for and renames them to the bare compound name: `new_names.get(name, name)` returns the new name where the dictionary has one, and the old name where it does not.

It then rebuilds the descriptor table the same way as last session, one `profile` dictionary per molecule and a column of [QED](https://www.nature.com/articles/nchem.1243) scores (line 5).

The column with [Lipinski's rule-of-five data](https://en.wikipedia.org/wiki/Lipinski%27s_rule_of_five) is left out this time, because none of today's questions need them.

In [ ]:
new_names = {"naproxen (S)": "naproxen", "ibuprofen (racemate)": "ibuprofen"}
dataset["name"] = [new_names.get(name, name) for name in dataset["name"]]

table = pd.DataFrame([profile(name, mol) for name, mol in zip(dataset["name"], dataset["mol"])])
table["qed"] = [round(QED.qed(mol), 3) for mol in dataset["mol"]]
table

### Two Molecules, One Weight

Last session's Part 3 warned that a descriptor is lossy which means that the information can describe multiple different things.
We used the example with [glucose](https://en.wikipedia.org/wiki/Glucose) and fructose.
One of our own compounds makes the same point: glucose is $\mathrm{C_6H_{12}O_6}$ and aspirin is $\mathrm{C_9H_8O_4}$, and both weigh 180.16 g/mol.
The glucose below is the ring form, written without stereochemistry to match the decision taken for the catalogue.

#### Question

2. Before running the block, predict which of the six numbers (molecular weight, logp, polar surface area, hydrogen bond donators and acceptors, and number of rotatable bonds) from the `profile` function will be the same for aspirin and glucose, and which will differ.


In [ ]:
aspirin = Chem.MolFromSmiles(supplier_catalogue["aspirin"])
glucose = Chem.MolFromSmiles("OCC1OC(O)C(O)C(O)C1O")
print("Aspirin:", profile("aspirin", aspirin))
print("Glucode:", profile("glucose", glucose))
Draw.MolsToGridImage([aspirin, glucose], legends=["aspirin", "glucose"], molsPerRow=2, subImgSize=(250, 200))

Six descriptors together can tell the two molecules apart, and even say in which properties they differ.
What they cannot say is what the two molecules *have in common*, atom for atom.
That is the question a fingerprint answers.

### What The MACCS Keys See

The [MACCS keys](https://doi.org/10.1021/ci010132r) from [video 6](https://youtu.be/Y6dmQ2TibIE) are 166 fixed true-or-false questions about substructures saved as either a 1 (true) or 0 (false).
Every molecule is asked the same 166 questions.
Because the list is fixed, every bit has a published meaning: bit 162, for example, asks whether the molecule has an aromatic atom, and bit 165 asks whether it has a ring atom.

#### Questions

3. Complete the two marked lines so that `shared` holds the on-bits the two molecules have in common and `either` the on-bits set in at least one of them. *Hint: the set operators `&` and `|` from track 1.*
4. Comment on what the results mean. Especially what we can use the *Fraction shared* number for.

In [ ]:
maccs_aspirin = MACCSkeys.GenMACCSKeys(aspirin)
maccs_glucose = MACCSkeys.GenMACCSKeys(glucose)
bits_aspirin = set(maccs_aspirin.GetOnBits())
bits_glucose = set(maccs_glucose.GetOnBits())
shared = bits_aspirin _ bits_glucose  #! on in *both* - the AND operator
either = bits_aspirin _ bits_glucose  #! on in *at least one* - the OR operator
print(f"Number of keys with True:\n  aspirin       = {len(bits_aspirin):d}\n  glucose       = {len(bits_glucose):d}\n  shared bits   = {len(shared):d}\n  either bits   = {len(either):d}")
print(f"Fraction shared = {len(shared) / len(either):.2f}")

#### Question

5. Predict the results for bits 162 (aromatic atoms in the molecule) and 165 (atoms in a ring) for both aspirin and then for glucose, before running it. *hint* see figure above here.


In [ ]:
print(f"bit 162 (aromatic atom):\n  aspirin = {maccs_aspirin.ToBitString()[162]}\n  glucose = {maccs_glucose.ToBitString()[162]}")
print(f"\nbit 165 (atoms in ring):\n  aspirin = {maccs_aspirin.ToBitString()[165]}\n  glucose = {maccs_glucose.ToBitString()[165]}")

#### Questions

6. 14 of the 35 bits set in either molecule are shared, a fraction of 0.40. What kind of question would both aspirin and glucose answer "yes" to, and is 0.40 a fair summary of how alike the two molecules are?

## Part 2 - Morgan Fingerprints For The Catalogue

A MACCS key can be thought of as a question a chemist wrote down in advance.

A Morgan fingerprint, [video 7](https://youtu.be/R4zspiNSmMY)'s [extended-connectivity fingerprint](https://doi.org/10.1021/ci100050t), asks no fixed questions but rather

>it grows a circular environment around every atom, one bond at a time, and sets one bit for every distinct environment it finds.

With `radius=2` an environment reaches *two bonds* out from its centre atom, roughly the size of a functional group, and that is the working default.

The generator is built once below with the code

```py
morgan_gen = AllChem.GetMorganGenerator(radius=2, fpSize=2048)
```

and reused for every molecule today, so the radius and the length can never drift between compounds.
Video 7's other rule was to look at the molecules before fingerprinting them.


In [ ]:
morgan_gen = AllChem.GetMorganGenerator(radius=2, fpSize=2048)
catalogue_names = list(dataset["name"])
catalogue_mols = list(dataset["mol"])
catalogue_fps = [morgan_gen.GetFingerprint(mol) for mol in catalogue_mols]
print(len(catalogue_fps), "fingerprints of", catalogue_fps[0].GetNumBits(), "bits each")
Draw.MolsToGridImage(catalogue_mols, legends=catalogue_names, molsPerRow=4, subImgSize=(220, 180))

The block below adds the heavy-atom count and the number of on-bits of every compound to `table`.

#### Question

1. Before running the block below, guess from the drawings which compound sets the most on-bits and which sets the fewest.


In [ ]:
table["heavy_atoms"] = [mol.GetNumAtoms() for mol in catalogue_mols]
table["on_bits"] = [fp.GetNumOnBits() for fp in catalogue_fps]
table

#### Question

2. Ketoprofen has two more heavy atoms than salbutamol, 19 against 17, and yet sets fewer on-bits, 28 against 33. How can the larger molecule set fewer bits?

### Tanimoto Similarity

[Video 8](https://youtu.be/iH6eMopWaio) put a single number on how alike two fingerprints are: the bits on in both ($A \cap B$), divided by the bits on in either of the two ($A \cup B$),

$$
T(A,B) = \frac{|A \cap B|}{|A \cup B|}
$$

which is exactly the fraction you computed by hand for aspirin and glucose in Part 1 above.
`DataStructs.TanimotoSimilarity` computes it for any two fingerprints (MACCS or Morgan) of the same kind.

Last session's Part 3 found that molecular weight, logP and TPSA cannot tell S-naproxen from R-naproxen.

#### Questions

3. Predict the Morgan Tanimoto similarity of S- and R-naproxen, using the two strings from the supplier's catalogue.
4. Predict whether the Morgan fingerprint will rate aspirin and glucose as more alike or less alike than the MACCS keys did. Then run the block and check both predictions.


In [ ]:
s_naproxen = Chem.MolFromSmiles(supplier_catalogue["naproxen (S)"])
r_naproxen = Chem.MolFromSmiles(supplier_catalogue["naproxen (R)"])
fp_s = morgan_gen.GetFingerprint(s_naproxen)
fp_r = morgan_gen.GetFingerprint(r_naproxen)
print("S- vs R-naproxen, Morgan:", round(DataStructs.TanimotoSimilarity(fp_s, fp_r), 3))
print("aspirin vs glucose, MACCS:", round(DataStructs.TanimotoSimilarity(maccs_aspirin, maccs_glucose), 3))
print("aspirin vs glucose, Morgan:", round(DataStructs.TanimotoSimilarity(morgan_gen.GetFingerprint(aspirin), morgan_gen.GetFingerprint(glucose)), 3))

### Which Of Our Compounds Look Alike?

Before judging anything new against the catalogue, it is worth knowing how alike the catalogue is to itself.
Video 8 compared every compound with the ones after it in the list, which visits each of the 21 pairs once.
The loop below does the same, but keeps the two names alongside every similarity so the pairs can be read afterwards.
**NB** the code below uses advanced slicing ([video 4 from Track 1](https://youtu.be/KLVQiM47GYY)) in lines 4 to 10 as well as a `tuple` data structure (also in video video 4 from Track 1) in line 18.

#### Question

5. Read the loop before running it. What does `row` hold for compound `i`, and why does `zip` pair every value in it with the right name from `catalogue_names[i+1:]`?


In [ ]:
all_pairs = []

for i in range(len(catalogue_fps)):
    # we extract the name of the molecule (this_name) and the fingerprint (this_fingerprint)
    this_name = catalogue_names[i]
    this_fingerprint = catalogue_fps[i]

    # we also extract all remaining molecules in the table (the [i+1:] syntax)
    other_names = catalogue_names[i+1:]
    other_fingerprints = catalogue_fps[i+1:]

    # compute all similarties between `this` and `other`.
    similarities = DataStructs.BulkTanimotoSimilarity(this_fingerprint, other_fingerprints)

    # store the data in all_pairs as a tuple
    for other_name, similarity in zip(other_names, similarities):
        similarity = round(similarity, 3)
        all_pairs.append((similarity, this_name, other_name))

# now sort all pairs according to similarity
# and print only the first 6 entries
ranked_pairs = sorted(all_pairs, reverse=True)
print(len(all_pairs), "pairs")
for pair in ranked_pairs[:6]:
    print(pair)

#### Answers

5. `row` holds the similarity of compound `i` to every compound after it, in list order, and `catalogue_names[i+1:]` is the names of exactly those compounds in the same order, so `zip` walks the two in step.
The similarity goes first in each tuple so that `sorted` ranks the pairs by it, as in video 8.

#### Questions

6. The three highest pairs are all drawn from naproxen, ibuprofen and ketoprofen. Look at the drawings from the start of this part: what do those three share that no other compound in the catalogue has?
7. Video 8 read a Morgan Tanimoto above 0.7 as close analogues, 0.4 to 0.7 as related scaffolds and below 0.3 as background. Where do the top three pairs fall, and where does everything else?

That is the result of the first half.
Our catalogue is one family of three profens, related at about 0.4, and four compounds that resemble nothing else in it.
Any new compound can now be placed against that picture, which is exactly what the supplier's next email is going to need.


## Part 3 - The Supplier's Second Offer

The supplier has offered nine more compounds.
Buying all of them would be easy, but several would mostly repeat information we already have from the compounds we own.
Those extras would not help the model learn anything new, and could even make the analysis noisier and harder to interpret.

Instead, the useful question is

> Which molecules from the new list of compounds adds chemistry the catalogue does not already have.

| offered compound | what the supplier claims |
|---|---|
| naproxen (R) | "naproxen", single enantiomer, cheaper than the S form |
| fenoprofen | an older profen anti-inflammatory |
| flurbiprofen | a profen, also sold in throat lozenges |
| [desloratadine](https://en.wikipedia.org/wiki/Desloratadine) | the active metabolite of loratadine, sold as an antihistamine in its own right |
| [terbutaline](https://en.wikipedia.org/wiki/Terbutaline) | a $\beta_2$-agonist bronchodilator, like salbutamol |
| salicylic acid | what aspirin becomes in the body |
| [diflunisal](https://en.wikipedia.org/wiki/Diflunisal) | a salicylate anti-inflammatory with a difluorophenyl ring |
| [celecoxib](https://en.wikipedia.org/wiki/Celecoxib) | a COX-2 selective anti-inflammatory |
| caffeine | a stimulant, and nothing like anything we hold |


In [ ]:
second_offer = {
    "naproxen (R)": "C[C@H](c1ccc2cc(OC)ccc2c1)C(=O)O",
    "fenoprofen": "CC(C(=O)O)c1cccc(Oc2ccccc2)c1",
    "flurbiprofen": "CC(C(=O)O)c1ccc(-c2ccccc2)c(F)c1",
    "desloratadine": "Clc1ccc2c(c1)CCc1cccnc1C2=C1CCNCC1",
    "terbutaline": "CC(C)(C)NCC(O)c1cc(O)cc(O)c1",
    "salicylic acid": "OC(=O)c1ccccc1O",
    "diflunisal": "OC(=O)c1cc(-c2ccc(F)cc2F)ccc1O",
    "celecoxib": "Cc1ccc(cc1)-c1cc(nn1-c1ccc(cc1)S(N)(=O)=O)C(F)(F)F",
    "caffeine": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
}

#### Question

1. The block below cleans the offer with the same helper and draws it. Before computing anything, compare the grid with the catalogue drawn at the start of Part 2 and write down, for each offered compound, which of our seven compounds you think it resembles most, or "none". Keep the list: you will check it against the fingerprint in a moment.


In [ ]:
offer = clean_catalogue(second_offer, keep_stereo=False)
Draw.MolsToGridImage(list(offer["mol"]), legends=list(offer["name"]), molsPerRow=3, subImgSize=(220, 180))

Finding the catalogue compound closest to an offered one is video 8's similarity search: one query, compared with a whole library by `BulkTanimotoSimilarity`, and the results ranked with `sorted`.
Here the library is our catalogue, and there are nine queries instead of one, so the search runs inside a loop.

#### Question

2. Complete the three marked lines inside the loop: fingerprint the offered compound with the same generator as the catalogue, compare it with every catalogue fingerprint, and take the best `(similarity, name)` pair from the top of the ranking.


In [ ]:
nearest_names = []
nearest_scores = []

for new_molecule in offer["mol"]:
    fp = morgan_gen.GetFingerprint(new_molecule)  #! the same generator as the catalogue
    similarities = DataStructs.BulkTanimotoSimilarity(fp, catalogue_fps)  #! one value per catalogue compound
    best_score, best_name = sorted(zip(similarities, catalogue_names), reverse=True)[0]  #! the top of the ranking
    nearest_names.append(best_name)
    nearest_scores.append(round(best_score, 3))

In [ ]:
offer["nearest"] = nearest_names
offer["tanimoto"] = nearest_scores
offer[["name", "nearest", "tanimoto"]].sort_values("tanimoto", ascending=False)

#### Question

3. Compare the table with your list from question 1. Where did you and the fingerprint agree, and which "nearest" compound would you not have called near at all?

Video 8's advice was to read a Tanimoto value against the structures, never on its own.
The block below draws every offered compound on the left and its nearest catalogue compound on the right, with the similarity in the legend.


In [ ]:
pair_mols = []
pair_legends = []
for name, mol, nearest, score in zip(offer["name"], offer["mol"], offer["nearest"], offer["tanimoto"]):
    pair_mols = pair_mols + [mol, catalogue_mols[catalogue_names.index(nearest)]]
    pair_legends = pair_legends + [name, nearest + " T=" + str(score)]

Draw.MolsToGridImage(pair_mols, legends=pair_legends, molsPerRow=2, subImgSize=(250, 180))

#### Questions

4. R-naproxen scores 1.000 against the naproxen we already hold. Is it therefore already in stock? Argue from last session's Part 2.
5. **Important:** Diflunisal is a salicylate anti-inflammatory, like aspirin, yet scores only 0.25 against it; celecoxib inhibits the same COX enzymes as the three profens and scores 0.12 against everything. What does that say about using a *high* similarity to find compounds with the same activity, and about using a *low* one to rule a compound out?

## Part 4: A Purchase Report And One Figure

The table from Part 3 is almost a report that we can use to make a purchase of new compounds.
However, a colleague reading it would have to know what a Tanimoto of 0.475 means.
So instead of presenting the Tanimoto values, we could instead print a label that makes more sense.

Video 8 gave the reading for Morgan fingerprints with radius 2 presented in the following table:

| Tanimoto | reading |
|---|---|
| above 0.7 | close analogues, often the same series |
| 0.4 to 0.7 | clearly related scaffolds |
| below 0.3 | background between unrelated drugs |

The implementation of the Tanimoto score to text is a function called `similarity_label` and is shown below:

In this exercise, a compound offered on the new list, but whose nearest catalogue compound is only at background similarity is new chemistry for the final catalogue.

#### Question

1. Read `similarity_label` before running it. It can return four answers, but video 8 gives only three readings: which answer is the extra one, which values produce it, and why does the function need it at all?


In [ ]:
def similarity_label(tanimoto):
    """Read a Morgan radius-2 Tanimoto value the way video 8 does."""
    if tanimoto > 0.7:
        return "close analogue"
    if tanimoto >= 0.4:
        return "related scaffold"
    if tanimoto < 0.3:
        return "background"
    return "between bands"

print(similarity_label(1.0), "|", similarity_label(0.583), "|", similarity_label(0.35), "|", similarity_label(0.123))

A new compound is only worth buying if it *also looks like something that could become a drug*.
[Last session exercise](https://colab.research.google.com/drive/1rfISwMt9RbOSogTfMQhQ4Yo-8nTCJmRo#scrollTo=b81654c4) gave you a number for that: [The quantitative estimate of drug-likeness](https://www.nature.com/articles/nchem.1243)(QED).
The `report` keeps the five columns a colleague needs, sorted by similarity; your part is to add the two columns it is built from and to save it.

#### Questions

2. Add a column `band` to `offer` by calling `similarity_band` on every value in the `tanimoto` column.
3. Add a column `qed` with the QED of every offered molecule, rounded to three decimals, the same way as the catalogue's `qed` column in Part 1.


In [ ]:
offer["band"] = [similarity_band(t) for t in offer["tanimoto"]]  #! one band per offered compound
offer["qed"] = [round(QED.qed(mol), 3) for mol in offer["mol"]]  #! drug-likeness, as last session
report = offer[["name", "nearest", "tanimoto", "band", "qed"]].sort_values("tanimoto", ascending=False)
report

### The Figure

The report has two numbers per compound that pull in different directions: similarity to what we already hold, where lower is newer, and QED, where higher is more drug-like.
A [scatter plot](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html) of one against the other shows the whole decision at once.
Each compound is drawn by its own `plt.scatter` call with a `label`, so the legend can name every point, and two dashed grey lines mark video 8's thresholds at 0.4 and 0.7.

#### Question

5. Complete the three marked lines: an x-axis label that names the fingerprint as well as the quantity, a y-axis label and a title.


In [ ]:
plt.figure(figsize=(8, 6))
for name, similarity, drug_likeness in zip(report["name"], report["tanimoto"], report["qed"]):
    plt.scatter(similarity, drug_likeness, s=80, label=name)

plt.plot([0.4, 0.4], [0, 1], linestyle="--", color="grey")
plt.plot([0.7, 0.7], [0, 1], linestyle="--", color="grey")
plt.xlim(0, 1.05)
plt.ylim(0, 1)
plt.xlabel("Tanimoto similarity to the nearest catalogue compound (Morgan, radius 2)")  #! the quantity and the fingerprint it was computed on
plt.ylabel("QED drug-likeness")  #! the y axis
plt.title("The second offer: novelty against drug-likeness")  #! a title
plt.legend(loc="lower right", fontsize=8)
plt.savefig("second_offer.png", dpi=300)
plt.show()

### A Shortlist

The figure shows the decision; a filter makes it exact.
QED has no official cut-off for "drug-like", so you have to choose one yourself, and the solution below uses 0.7.

#### Question

6. Write one line that keeps only the rows of `report` in the background band with a QED above your cut-off, and store them as `shortlist`. Then say in one sentence why you chose that cut-off. *Hint: a combined boolean mask with `&`, like last session's Lipinski filter.*


In [ ]:
shortlist = report[(report["band"] == "background") & (report["qed"] > 0.7)]  #! background band and above your QED cut-off
shortlist

#### Answers

6. With 0.7 the shortlist is diflunisal and celecoxib.
The cut-off is a decision, not a result: lower it to 0.5 and caffeine, at 0.538, joins them, while 0.8 leaves diflunisal alone.
One way to argue for 0.7 is that four of our own seven compounds pass it, so a compound bought for the catalogue should look at least as drug-like as most of what is already there.

#### Questions

7. Using the shortlist and both axes of the figure, which compounds would you recommend buying to broaden the catalogue, and which would you leave with the supplier? Discuss it with your neighbour (or yourself): there is more than one defensible list.
8. Only one compound landed in the close-analogue band, and it is the one compound a pharmacologist would never call a duplicate of anything we hold. What would you add to the report so that it cannot slip through a second time? *Hint: last session's Part 2.*

That is today's result.
Seven compounds from last session and nine from the new offer have become a saved report and one figure: for every offered compound its nearest relative in the catalogue, how near that relative is in words as well as numbers, and how drug-like the compound looks.

Both halves of the report were computed from connectivity alone.
Neither the fingerprint nor QED knows anything about stereochemistry or activity, and the two places where that mattered today, R-naproxen and celecoxib, are exactly the two a chemist had to catch by hand.
